# Code quality
<img src='../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

Writing modular, reusable code is important for code quality.

Code is a means to communicate: You use it to communicate with machines but also with other developers. Therefore high quality code is good communication.

Code of high quality is correct, human readable, consistent, modular and reusable.

In this notebook, you will practice refactoring code to improve its quality (e.g. how easy is to understand, modify, maintain, test). 

<a id='refactor'></a>
## Refactoring

The code in `add_features()` produces the correct output, but it's not good code (yet).
The function is doing multiple things (checking sex, getting hair type, etc.) and that is [not OK](https://blog.codinghorror.com/curlys-law-do-one-thing/).

### <mark> Exercise: Refactoring

Move the sub-logic from `add_features()`  to the appropriate functions into separate functions:

 - `check_has_name()`
 - `get_sex()`
 - `get_neutered()`
 - `get_hair_type()`
 - `compute_days_upon_outcome()`    

 The function `check_is_dog()` is already filled in for you.
 All functions take a `Series` (a column in our `DataFrame`) and return a `Series`.

After this exercise, the `add_features()` function should look something like:


 ```python
 def add_features(df):
     df['is_dog'] = check_is_dog(df['animal_type'])
     df['has_name'] = check_has_name(df['name'])
     # ...
     return df
 ```

You can use the following documented function definitions.

In [ ]:

import pandas as pd
import numpy as np


def check_has_name(df):
    """Check if the animal is not called 'unknown'.
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the animal data including 'name' column.
    Returns
    -------
    df : pandas.DataFrame
        DataFrame containing the animal data with the 'has_name' column added.
    """
    
    df['has_name'] = df['name'].str.lower() != 'unknown'
    
    return df


def get_sex(df):
    """Determine if the sex was 'Male', 'Female' or unknown.
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the animal data including 'sex_upon_outcome' column.
    Returns
    -------
    df : pandas.DataFrame
        DataFrame containing the animal data with the 'sex' column added.
    """
    
    sexUponOutcome = df['sex_upon_outcome']
    sex = pd.Series('unknown', index=sexUponOutcome.index)

    sex.loc[sexUponOutcome.str.endswith('Female')] = 'female'
    sex.loc[sexUponOutcome.str.endswith('Male')] = 'male'
    df['sex'] = sex
    
    return df


def get_neutered(df):
    """Determine if an animal was intact or not.
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the animal data including 'sex_upon_outcome' column.
    Returns
    -------
    df : pandas.DataFrame
        DataFrame containing the animal data with the 'neutered' column added.  
    """
    
    sexUponOutcome = df['sex_upon_outcome']
    neutered = sexUponOutcome.str.lower()
    neutered.loc[neutered.str.contains('neutered')] = 'fixed'
    neutered.loc[neutered.str.contains('spayed')] = 'fixed'


    neutered.loc[neutered.str.contains('intact')] = 'intact'
    neutered.loc[~neutered.isin(['fixed', 'intact'])] = 'unknown'


    # df['neutered'] = neutered
    
    return df['neutered']


def get_hair_type(df):
    """Get hair type of a breed.
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the animal data including 'breed' column.  
    Returns
    -------
    df : pandas.DataFrame
        DataFrame containing the animal data with the 'hair_type' column added.
    """
    
    hairType = df['breed'].str.lower()
    Valid_hair_types = ['shorthair', 'medium hair', 'longhair']

    for hair in Valid_hair_types:
        is_hair_type = hairType.str.contains(hair)
        hairType[is_hair_type] = hair

    hairType[~hairType.isin(Valid_hair_types)] = 'unknown'


    df['hair_type'] = hairType
    
    return df 


def compute_days_upon_outcome(df):
    """Compute age in days upon outcome.
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the animal data including 'age_upon_outcome' column.
    Returns
    -------
    days_upon_outcome : pandas.Series
        Age in days
    """
    
    Split_Age = df['age_upon_outcome'].str.split()
    time = Split_Age.apply(lambda x: x[0] if x[0] != 'Unknown' else np.nan)
    period = Split_Age.apply(lambda x: x[1] if x[0] != 'Unknown' else None)
    period_Mapping = {'year': 365, 'years': 365, 'weeks': 7, 'week': 7,
                      'month': 30, 'months': 30, 'days': 1, 'day': 1}
    days_upon_outcome = time.astype(float) * period.map(period_Mapping)
    df['days_upon_outcome'] = days_upon_outcome
    
    return df


You can run the following cells to test your changes while developing:

In [1]:
from animal_shelter.data import load_data
from animal_shelter.features import add_features
animal_outcomes = load_data('../data/test.csv')
with_features = add_features(animal_outcomes)
with_features.head(20)

,id,name,date_time,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,is_dog,has_name,sex,neutered,hair_type,days_upon_outcome
0,1,Summer,2015-10-12 12:15:00,Dog,Intact Female,10 months,Labrador Retriever Mix,Red/White,True,True,female,intact,unknown,300.0
1,2,Cheyenne,2014-07-26 17:59:00,Dog,Spayed Female,2 years,German Shepherd/Siberian Husky,Black/Tan,True,True,female,fixed,unknown,730.0
2,3,Gus,2016-01-13 12:20:00,Cat,Neutered Male,1 year,Domestic Shorthair Mix,Brown Tabby,False,True,male,fixed,shorthair,365.0
3,4,Pongo,2013-12-28 18:12:00,Dog,Intact Male,4 months,Collie Smooth Mix,Tricolor,True,True,male,intact,unknown,120.0
4,5,Skooter,2015-09-24 17:59:00,Dog,Neutered Male,2 years,Miniature Poodle Mix,White,True,True,male,fixed,unknown,730.0
5,6,Beau,2015-06-23 11:17:00,Dog,Neutered Male,3 years,Beagle Mix,Brown/White,True,True,male,fixed,unknown,1095.0
6,7,Bobo,2014-03-12 09:45:00,Cat,Neutered Male,13 years,Domestic Medium Hair Mix,Brown Tabby/White,False,True,male,fixed,medium hair,4745.0
7,8,Abby,2014-06-25 08:27:00,Cat,Spayed Female,6 months,Domestic Shorthair Mix,Brown Tabby,False,True,female,fixed,shorthair,180.0
8,9,Ruby Grace,2014-11-12 18:05:00,Dog,Spayed Female,3 months,Cairn Terrier,Black/Cream,True,True,female,fixed,unknown,90.0
9,10,Ruby,2014-04-07 17:41:00,Dog,Spayed Female,1 year,Pit Bull Mix,Brown/White,True,True,female,fixed,unknown,365.0


In [2]:
%load_ext autoreload
%autoreload 2

### <mark> Exercise:</mark> Side effects

It already looks better and more structured, but there are still things that should be improved.

 The function `add_features()` has an unexpected [side effect](https://softwareengineering.stackexchange.com/questions/15269/why-are-side-effects-considered-evil-in-functional-programming): the input `df` gets changed when the function is called.
    
 Generally, you want to avoid this kind of behaviour. How could you avoid this?
 
 You could use `.copy()` to create a copy of the object, or use the `pd.DataFrame.assign()` method.
 

 What would you do to improve these functions further?

### <mark> Exercise:</mark> Styling

Through the refactor you might notice that the style of some of the code might no adhere to PEP8 guidelines (e.g. using snake_case for variable names). Feel free to change them if you notice some errors, but in the next notebook you will see how to automatically detect style violations.

Additionally, in Python modules it's common to denote functions that are only meant to be used from within the module with an underscore. For example, in the `features` module, `load_data` is a function that other modules might import an use, but all other functions are likely just gonna be used from within the module. You can mark them as internal with a leading underscore.